# PhotoMappers Lifeline Reasoning Analysis

**Author:** NUS PhotoMapper Team  
**Date:** 2026-09-24

## Objective
Analyze VLM geographic reasoning performance across FEMA Community Lifelines with consistent handling of missing values and reporting rules.

## Inputs
- `data/input/reasoning_lifeline_20260918.xlsx`

## Outputs
- `data/outputs/lifeline_reasoning_long_cleaned.csv`
- `data/outputs/lifeline_data_availability.csv`
- `data/outputs/lifeline_overall_performance.csv`
- `data/outputs/lifeline_geographic_cue.csv`
- `data/outputs/lifeline_reasoning_evidence.csv`
- Publication figures exported to `data/outputs/`

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from photomapper_common import (
    detect_columns,
    detect_header_row,
    geographic_cue_from_indicator,
    reasoning_evidence_from_indicator,
)

warnings.filterwarnings('ignore', category=UserWarning)

# Paths
PROJECT_DIR = Path.cwd()
INPUT_XLSX = PROJECT_DIR / 'data' / 'input' / 'reasoning_lifeline_20260918.xlsx'
OUTPUT_DIR = PROJECT_DIR / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_XLSX.exists():
    raise FileNotFoundError(f'Input workbook not found: {INPUT_XLSX}')

# Lifeline order and harmonization
LIFELINE_ORDER = [
    'Safety and Security',
    'Food, Hydration, Shelter',
    'Health and Medical',
    'Energy',
    'Communications',
    'Transportation',
    'Hazardous Materials',
    'Water Systems',
]

LIFELINE_SHEET_TO_NAME = {
    'Safety_Security': 'Safety and Security',
    'Food_Water_Shelter': 'Food, Hydration, Shelter',
    'Health_Medical': 'Health and Medical',
    'Energy': 'Energy',
    'Communications': 'Communications',
    'Transportation': 'Transportation',
    'Hazmat': 'Hazardous Materials',
    'Water_Systems': 'Water Systems',
    'Other': 'Other',
}

GEOGRAPHIC_CUE_ORDER = [
    'Building and structure',
    'Road and transport',
    'Vegetation and terrain',
    'Objects and facilities',
    'Global scene',
]

REASONING_EVIDENCE_ORDER = [
    'Appearance',
    'Spatial layout',
    'Landmark cue',
]

MODEL_ORDER = ['Qwen3.6-27B', 'InternVL3-8B']

MODEL_COLORS = {
    'Qwen3.6-27B': '#2166AC',
    'InternVL3-8B': '#B2182B',
}

EVIDENCE_COLORS = {
    'Appearance': '#9ECAE1',
    'Spatial layout': '#3182BD',
    'Landmark cue': '#08519C',
}

CUE_COLORS = {
    'Building and structure': '#2166AC',
    'Road and transport': '#67A9CF',
    'Vegetation and terrain': '#74C476',
    'Objects and facilities': '#238B45',
    'Global scene': '#006D2C',
}

CUE_MARKERS = {
    'Building and structure': 'o',
    'Road and transport': '^',
    'Vegetation and terrain': 's',
    'Objects and facilities': 'D',
    'Global scene': 'P',
}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.linewidth': 0.8,
})

print(f'Project directory: {PROJECT_DIR}')
print(f'Input workbook: {INPUT_XLSX}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Helper functions for robust worksheet parsing
def parse_lifeline_sheet(df_raw, sheet_name):
    header_idx = detect_header_row(df_raw)
    col_map_raw = detect_columns(df_raw.iloc[header_idx].tolist())

    col_map = {
        'no': col_map_raw['no_col'],
        'indicator': col_map_raw['indicator_col'],
        'q_kappa': col_map_raw['Qwen3.6-27B_kappa'],
        'q_agreement': col_map_raw['Qwen3.6-27B_agr'],
        'i_kappa': col_map_raw['InternVL3-8B_kappa'],
        'i_agreement': col_map_raw['InternVL3-8B_agr'],
    }

    body = df_raw.iloc[header_idx + 1:].copy()
    body.columns = df_raw.iloc[header_idx].tolist()
    body = body.dropna(how='all')

    data_rows = []
    for _, row in body.iterrows():
        no_val = pd.to_numeric(row[col_map['no']], errors='coerce')
        if pd.isna(no_val):
            continue
        if int(no_val) < 1 or int(no_val) > 30:
            continue

        n = int(no_val)
        indicator = row[col_map['indicator']]

        record = {
            'Sheet': sheet_name,
            'Lifeline': LIFELINE_SHEET_TO_NAME.get(sheet_name, sheet_name),
            'Indicator_No': n,
            'Indicator': str(indicator).strip() if pd.notna(indicator) else f'Indicator {n}',
            'Geographic_Cue': geographic_cue_from_indicator(n),
            'Reasoning_Evidence': reasoning_evidence_from_indicator(n),
            'Qwen3.6-27B_kappa_w': pd.to_numeric(row[col_map['q_kappa']], errors='coerce'),
            'Qwen3.6-27B_agreement': pd.to_numeric(row[col_map['q_agreement']], errors='coerce'),
            'InternVL3-8B_kappa_w': pd.to_numeric(row[col_map['i_kappa']], errors='coerce'),
            'InternVL3-8B_agreement': pd.to_numeric(row[col_map['i_agreement']], errors='coerce'),
        }
        data_rows.append(record)

    out = pd.DataFrame(data_rows)
    return out, header_idx, col_map

In [ ]:
# 1) Inspect workbook structure and parse indicator-level data
xls = pd.ExcelFile(INPUT_XLSX)
detected_sheets = xls.sheet_names

inspection_rows = []
parsed_by_sheet = {}
errors_by_sheet = {}

for s in detected_sheets:
    raw = pd.read_excel(INPUT_XLSX, sheet_name=s, header=None)
    non_empty_rows = int(raw.notna().any(axis=1).sum())
    try:
        parsed, header_idx, col_map = parse_lifeline_sheet(raw, s)
        parsed_by_sheet[s] = parsed
        inspection_rows.append({
            'Sheet': s,
            'Mapped_Lifeline': LIFELINE_SHEET_TO_NAME.get(s, s),
            'Rows': int(raw.shape[0]),
            'Columns': int(raw.shape[1]),
            'Non_Empty_Rows': non_empty_rows,
            'Header_Row_Index': header_idx,
            'Indicators_Detected': int(parsed['Indicator_No'].nunique()),
            'Qwen_kappa_col': col_map['q_kappa'],
            'Qwen_agreement_col': col_map['q_agreement'],
            'Intern_kappa_col': col_map['i_kappa'],
            'Intern_agreement_col': col_map['i_agreement'],
        })
    except Exception as e:
        errors_by_sheet[s] = str(e)
        inspection_rows.append({
            'Sheet': s,
            'Mapped_Lifeline': LIFELINE_SHEET_TO_NAME.get(s, s),
            'Rows': int(raw.shape[0]),
            'Columns': int(raw.shape[1]),
            'Non_Empty_Rows': non_empty_rows,
            'Header_Row_Index': np.nan,
            'Indicators_Detected': 0,
            'Qwen_kappa_col': np.nan,
            'Qwen_agreement_col': np.nan,
            'Intern_kappa_col': np.nan,
            'Intern_agreement_col': np.nan,
        })

inspection_df = pd.DataFrame(inspection_rows).sort_values('Sheet').reset_index(drop=True)
print('Detected worksheets:')
print(detected_sheets)
print('\nWorkbook structure inspection:')
display(inspection_df)

if errors_by_sheet:
    print('\nSheets with parsing errors:')
    for k, v in errors_by_sheet.items():
        print(f'- {k}: {v}')

if not parsed_by_sheet:
    raise RuntimeError('No sheets were parsed successfully.')

all_indicator_df = pd.concat(parsed_by_sheet.values(), ignore_index=True)

# Keep original parsed long data (includes Other)
long_df = all_indicator_df.melt(
    id_vars=['Sheet', 'Lifeline', 'Indicator_No', 'Indicator', 'Geographic_Cue', 'Reasoning_Evidence'],
    value_vars=[
        'Qwen3.6-27B_kappa_w',
        'Qwen3.6-27B_agreement',
        'InternVL3-8B_kappa_w',
        'InternVL3-8B_agreement',
    ],
    var_name='ModelMetric',
    value_name='Value',
)

long_df['Model'] = np.where(long_df['ModelMetric'].str.startswith('Qwen3.6-27B'), 'Qwen3.6-27B', 'InternVL3-8B')
long_df['Metric'] = np.where(long_df['ModelMetric'].str.endswith('_kappa_w'), 'kappa_w', 'agreement_rate')
long_df = long_df.drop(columns=['ModelMetric'])

long_df = long_df.pivot_table(
    index=['Sheet', 'Lifeline', 'Indicator_No', 'Indicator', 'Geographic_Cue', 'Reasoning_Evidence', 'Model'],
    columns='Metric',
    values='Value',
    aggfunc='first'
).reset_index()
long_df.columns.name = None

long_df['kappa_w'] = pd.to_numeric(long_df['kappa_w'], errors='coerce')
long_df['agreement_rate'] = pd.to_numeric(long_df['agreement_rate'], errors='coerce')
long_df['Lifeline_In_Main8'] = long_df['Lifeline'].isin(LIFELINE_ORDER)

long_df = long_df.sort_values(['Lifeline', 'Model', 'Indicator_No']).reset_index(drop=True)

clean_long_path = OUTPUT_DIR / 'lifeline_reasoning_long_cleaned.csv'
long_df.to_csv(clean_long_path, index=False)

print(f'\nSaved cleaned long-format CSV: {clean_long_path}')
display(long_df.head(12))

In [ ]:
# 2) Validation and data availability summary
main8_df = long_df[long_df['Lifeline'].isin(LIFELINE_ORDER)].copy()
main8_df['Lifeline'] = pd.Categorical(main8_df['Lifeline'], categories=LIFELINE_ORDER, ordered=True)

# Worksheet emptiness based on parse and metric definition
sheet_to_lifeline = {k: v for k, v in LIFELINE_SHEET_TO_NAME.items() if k in detected_sheets}
empty_sheets = []
for s, lifeline in sheet_to_lifeline.items():
    sdf = long_df[long_df['Sheet'] == s]
    if sdf.empty:
        empty_sheets.append(s)

# Missing indicator checks (expect 30 indicators for each model/lifeline where data rows exist)
indicator_counts = main8_df.groupby(['Lifeline', 'Model'])['Indicator_No'].nunique().reset_index(name='N_indicators_detected')

# Undefined/missing kappa distinction from genuine zeros
kappa_missing_counts = main8_df.groupby(['Lifeline', 'Model'])['kappa_w'].apply(lambda s: int(s.isna().sum())).reset_index(name='N_missing_kappa')
kappa_zero_counts = main8_df.groupby(['Lifeline', 'Model'])['kappa_w'].apply(lambda s: int((s == 0).sum())).reset_index(name='N_zero_kappa')

availability = main8_df.groupby(['Lifeline', 'Model'], as_index=False).agg(
    N_valid_kappa=('kappa_w', lambda s: int(s.notna().sum())),
    N_valid_agreement=('agreement_rate', lambda s: int(s.notna().sum())),
)

def _availability_status(row):
    nk = row['N_valid_kappa']
    na = row['N_valid_agreement']
    if nk == 0 and na == 0:
        return 'No valid observations'
    if nk < 30 or na < 30:
        return 'Partially available'
    return 'Valid'

availability['Status'] = availability.apply(_availability_status, axis=1)
availability['Lifeline'] = pd.Categorical(availability['Lifeline'], categories=LIFELINE_ORDER, ordered=True)
availability = availability.sort_values(['Lifeline', 'Model']).reset_index(drop=True)

availability_path = OUTPUT_DIR / 'lifeline_data_availability.csv'
availability.to_csv(availability_path, index=False)

# Additional validation checks
missing_indicator_rows = indicator_counts[indicator_counts['N_indicators_detected'] != 30].copy()
lifelines_present_final = sorted(main8_df['Lifeline'].dropna().astype(str).unique().tolist())
missing_lifelines_final = sorted(set(LIFELINE_ORDER) - set(lifelines_present_final))

print('Validation summary')
print('------------------')
print(f'Empty worksheets detected: {empty_sheets if empty_sheets else "None"}')
print(f'Parsed sheets with errors: {list(errors_by_sheet.keys()) if errors_by_sheet else "None"}')
print(f'Missing indicator sets (not equal to 30): {len(missing_indicator_rows)} combinations')
if len(missing_indicator_rows) > 0:
    display(missing_indicator_rows.sort_values(['Lifeline', 'Model']))
print(f"Undefined kappa (NaN) total in main 8: {int(main8_df['kappa_w'].isna().sum())}")
print(f"Genuine kappa zeros total in main 8: {int((main8_df['kappa_w'] == 0).sum())}")
print(f'All 8 lifelines retained in final data: {len(missing_lifelines_final) == 0}')
if missing_lifelines_final:
    print(f'Missing lifelines in final data: {missing_lifelines_final}')

print(f'\nSaved: {availability_path}')
display(availability)

# Keep additional tables for transparent zero-vs-NaN audit
zero_missing_audit = availability[['Lifeline', 'Model', 'N_valid_kappa']].merge(kappa_missing_counts, on=['Lifeline', 'Model']).merge(kappa_zero_counts, on=['Lifeline', 'Model'])
display(zero_missing_audit.sort_values(['Lifeline', 'Model']))

In [ ]:
# 3) Aggregate performance (from indicator-level values only)
overall_perf = main8_df.groupby(['Lifeline', 'Model'], as_index=False).agg(
    Mean_kappa=('kappa_w', 'mean'),
    Mean_Agreement=('agreement_rate', 'mean'),
    N_valid=('kappa_w', lambda s: int(s.notna().sum())),
)

# Ensure all Lifeline x Model combinations are present and preserve NaN for missing
all_pairs = pd.MultiIndex.from_product([LIFELINE_ORDER, MODEL_ORDER], names=['Lifeline', 'Model']).to_frame(index=False)
overall_perf = all_pairs.merge(overall_perf, on=['Lifeline', 'Model'], how='left')
overall_perf['N_valid'] = overall_perf['N_valid'].fillna(0).astype(int)

geo_perf = main8_df.groupby(['Lifeline', 'Model', 'Geographic_Cue'], as_index=False).agg(
    Mean_kappa=('kappa_w', 'mean'),
    Mean_Agreement=('agreement_rate', 'mean'),
)
geo_pairs = pd.MultiIndex.from_product([LIFELINE_ORDER, MODEL_ORDER, GEOGRAPHIC_CUE_ORDER], names=['Lifeline', 'Model', 'Geographic Cue']).to_frame(index=False)
geo_perf = geo_perf.rename(columns={'Geographic_Cue': 'Geographic Cue'})
geo_perf = geo_pairs.merge(geo_perf, on=['Lifeline', 'Model', 'Geographic Cue'], how='left')

evidence_perf = main8_df.groupby(['Lifeline', 'Model', 'Reasoning_Evidence'], as_index=False).agg(
    Mean_kappa=('kappa_w', 'mean'),
    Mean_Agreement=('agreement_rate', 'mean'),
)
ev_pairs = pd.MultiIndex.from_product([LIFELINE_ORDER, MODEL_ORDER, REASONING_EVIDENCE_ORDER], names=['Lifeline', 'Model', 'Reasoning Evidence']).to_frame(index=False)
evidence_perf = evidence_perf.rename(columns={'Reasoning_Evidence': 'Reasoning Evidence'})
evidence_perf = ev_pairs.merge(evidence_perf, on=['Lifeline', 'Model', 'Reasoning Evidence'], how='left')

overall_path = OUTPUT_DIR / 'lifeline_overall_performance.csv'
geo_path = OUTPUT_DIR / 'lifeline_geographic_cue.csv'
ev_path = OUTPUT_DIR / 'lifeline_reasoning_evidence.csv'

overall_perf.to_csv(overall_path, index=False)
geo_perf.to_csv(geo_path, index=False)
evidence_perf.to_csv(ev_path, index=False)

print(f'Saved: {overall_path}')
print(f'Saved: {geo_path}')
print(f'Saved: {ev_path}')

display(overall_perf)

In [ ]:
# Utility for consistent axis style
def style_axis(ax, add_grid=True):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    if add_grid:
        ax.grid(axis='y', linestyle='-', linewidth=0.4, alpha=0.2)

def consistent_ylim(*series_list):
    vals = pd.concat([pd.Series(s).dropna() for s in series_list if s is not None], ignore_index=True)
    if vals.empty:
        return (0.0, 1.0)
    ymin = max(0.0, float(vals.min()) - 0.03)
    ymax = min(1.0, float(vals.max()) + 0.03)
    if ymin >= ymax:
        ymin, ymax = 0.0, 1.0
    return ymin, ymax

In [ ]:
# 4) Figure 1: Overall performance dumbbell plot (kappa_w)
fig1_df = overall_perf.copy()
fig1_df['Lifeline'] = pd.Categorical(fig1_df['Lifeline'], categories=LIFELINE_ORDER, ordered=True)
pivot_k = fig1_df.pivot(index='Lifeline', columns='Model', values='Mean_kappa').reindex(LIFELINE_ORDER)

fig, ax = plt.subplots(figsize=(6.8, 4.6), constrained_layout=True)

ypos = np.arange(len(LIFELINE_ORDER))
for i, lf in enumerate(LIFELINE_ORDER):
    qv = pivot_k.loc[lf, 'Qwen3.6-27B'] if 'Qwen3.6-27B' in pivot_k.columns else np.nan
    iv = pivot_k.loc[lf, 'InternVL3-8B'] if 'InternVL3-8B' in pivot_k.columns else np.nan

    if pd.notna(qv) and pd.notna(iv):
        ax.plot([qv, iv], [i, i], color='#8A8A8A', linewidth=1.2, zorder=1)

    if pd.notna(qv):
        ax.scatter(qv, i, color=MODEL_COLORS['Qwen3.6-27B'], s=28, zorder=3, label='Qwen3.6-27B' if i == 0 else None)
    if pd.notna(iv):
        ax.scatter(iv, i, color=MODEL_COLORS['InternVL3-8B'], s=28, zorder=3, label='InternVL3-8B' if i == 0 else None)

ax.set_yticks(ypos)
ax.set_yticklabels(LIFELINE_ORDER)
ax.invert_yaxis()
ax.set_xlabel('Mean quadratic weighted kappa (kappa_w)')
ax.set_ylabel('Community Lifeline')

xlim = consistent_ylim(pivot_k.get('Qwen3.6-27B', pd.Series(dtype=float)), pivot_k.get('InternVL3-8B', pd.Series(dtype=float)))
ax.set_xlim(xlim)

style_axis(ax, add_grid=False)
ax.grid(axis='x', linestyle='-', linewidth=0.4, alpha=0.2)
ax.legend(frameon=False, ncol=2, loc='lower right', handletextpad=0.4, columnspacing=0.8)

fig1_path = OUTPUT_DIR / 'Fig1_Lifeline_Overall_Kappa.png'
fig.savefig(fig1_path, dpi=600, transparent=True, bbox_inches='tight')
plt.show()

print(f'Saved: {fig1_path}')

In [ ]:
# 5) Figure 2: Reasoning Evidence by Lifeline (two-panel line figure)
fig2_df = evidence_perf.copy()
fig2_df['Lifeline'] = pd.Categorical(fig2_df['Lifeline'], categories=LIFELINE_ORDER, ordered=True)
fig2_df['Reasoning Evidence'] = pd.Categorical(fig2_df['Reasoning Evidence'], categories=REASONING_EVIDENCE_ORDER, ordered=True)

ymin, ymax = consistent_ylim(fig2_df['Mean_kappa'])
x = np.arange(len(LIFELINE_ORDER))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.6), sharey=True, constrained_layout=True)

for ax, model in zip(axes, MODEL_ORDER):
    mdf = fig2_df[fig2_df['Model'] == model].copy()
    for ev in REASONING_EVIDENCE_ORDER:
        sdf = mdf[mdf['Reasoning Evidence'] == ev].sort_values('Lifeline')
        y = sdf.set_index('Lifeline').reindex(LIFELINE_ORDER)['Mean_kappa']
        ax.plot(x, y.values, marker='o', markersize=4, linewidth=1.9, color=EVIDENCE_COLORS[ev], label=ev)

    ax.set_title(model)
    ax.set_xticks(x)
    ax.set_xticklabels(LIFELINE_ORDER, rotation=40, ha='right')
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel('Community Lifeline')
    style_axis(ax, add_grid=True)

axes[0].set_ylabel('Mean quadratic weighted kappa (kappa_w)')
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles, labels, frameon=False, loc='lower right')

fig2_path = OUTPUT_DIR / 'Fig2_Lifeline_Reasoning_Evidence.png'
fig.savefig(fig2_path, dpi=600, transparent=True, bbox_inches='tight')
plt.show()

print(f'Saved: {fig2_path}')

In [ ]:
# 6) Figure 3: Geographic Cue by Lifeline (two-panel line figure)
fig3_df = geo_perf.copy()
fig3_df['Lifeline'] = pd.Categorical(fig3_df['Lifeline'], categories=LIFELINE_ORDER, ordered=True)
fig3_df['Geographic Cue'] = pd.Categorical(fig3_df['Geographic Cue'], categories=GEOGRAPHIC_CUE_ORDER, ordered=True)

ymin, ymax = consistent_ylim(fig3_df['Mean_kappa'])
x = np.arange(len(LIFELINE_ORDER))

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.8), sharey=True, constrained_layout=True)

for ax, model in zip(axes, MODEL_ORDER):
    mdf = fig3_df[fig3_df['Model'] == model].copy()
    for cue in GEOGRAPHIC_CUE_ORDER:
        sdf = mdf[mdf['Geographic Cue'] == cue].sort_values('Lifeline')
        y = sdf.set_index('Lifeline').reindex(LIFELINE_ORDER)['Mean_kappa']
        ax.plot(
            x, y.values,
            marker=CUE_MARKERS[cue], markersize=4, linewidth=1.9,
            color=CUE_COLORS[cue], label=cue
        )

    ax.set_title(model)
    ax.set_xticks(x)
    ax.set_xticklabels(LIFELINE_ORDER, rotation=40, ha='right')
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel('Community Lifeline')
    style_axis(ax, add_grid=True)

axes[0].set_ylabel('Mean quadratic weighted kappa (kappa_w)')
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles, labels, frameon=False, loc='lower right', ncol=1)

fig3_path = OUTPUT_DIR / 'Fig3_Lifeline_Geographic_Cues.png'
fig.savefig(fig3_path, dpi=600, transparent=True, bbox_inches='tight')
plt.show()

print(f'Saved: {fig3_path}')

In [ ]:
# 7) Figure 4: Indicator-level heatmaps (kappa_w + agreement)
heat_df = main8_df.copy()
heat_df['Lifeline'] = pd.Categorical(heat_df['Lifeline'], categories=LIFELINE_ORDER, ordered=True)

# User-provided abbreviations (main 8 only, ordered to match LIFELINE_ORDER).
LIFELINE_ABBR_MAP = {
    'Safety and Security': 'SS',
    'Food, Hydration, Shelter': 'FWS',
    'Health and Medical': 'HM',
    'Energy': 'ENER',
    'Communications': 'COMM',
    'Transportation': 'TRAN',
    'Hazardous Materials': 'HAZM',
    'Water Systems': 'WS',
}
LIFELINE_ABBR = [LIFELINE_ABBR_MAP[l] for l in LIFELINE_ORDER]

indicator_order_df = heat_df[['Indicator_No', 'Indicator']].drop_duplicates().sort_values('Indicator_No')
indicator_order = indicator_order_df['Indicator'].tolist()

model_mats_kappa = {}
model_mats_agree = {}
for model in MODEL_ORDER:
    m = heat_df[heat_df['Model'] == model].copy()
    mat_k = m.pivot_table(index='Indicator', columns='Lifeline', values='kappa_w', aggfunc='first')
    mat_a = m.pivot_table(index='Indicator', columns='Lifeline', values='agreement_rate', aggfunc='first')
    mat_k = mat_k.reindex(index=indicator_order, columns=LIFELINE_ORDER)
    mat_a = mat_a.reindex(index=indicator_order, columns=LIFELINE_ORDER)
    model_mats_kappa[model] = mat_k
    model_mats_agree[model] = mat_a

all_kappa = pd.concat([model_mats_kappa[m].stack().dropna() for m in MODEL_ORDER], ignore_index=True)
if all_kappa.empty:
    vmin_k, vmax_k = 0.0, 1.0
else:
    vmin_k = max(0.0, float(all_kappa.min()))
    vmax_k = min(1.0, float(all_kappa.max()))
    if vmin_k == vmax_k:
        vmax_k = min(1.0, vmin_k + 0.1)

all_agree_pct = pd.concat([(model_mats_agree[m] * 100.0).stack().dropna() for m in MODEL_ORDER], ignore_index=True)
if all_agree_pct.empty:
    vmin_a, vmax_a = 0.0, 100.0
else:
    vmin_a = max(0.0, float(all_agree_pct.min()))
    vmax_a = min(100.0, float(all_agree_pct.max()))
    if vmin_a == vmax_a:
        vmax_a = min(100.0, vmin_a + 1.0)

cmap_kappa = plt.get_cmap('Blues').copy()
cmap_kappa.set_bad('#E6E6E6')

# Similar agreement palette direction as 04 (sequential red family), while keeping missing values neutral.
cmap_agree = plt.get_cmap('Reds').copy()
cmap_agree.set_bad('#E6E6E6')

def _plot_heatmap_pair(model_mats, out_name_q, out_name_i, cmap, vmin, vmax, cbar_label):
    for model, out_name in [('Qwen3.6-27B', out_name_q), ('InternVL3-8B', out_name_i)]:
        mat = model_mats[model]
        mat_masked = np.ma.masked_invalid(mat.to_numpy(dtype=float))

        fig, ax = plt.subplots(figsize=(10.2, 9.0))
        im = ax.imshow(mat_masked, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')

        ax.set_xticks(np.arange(len(LIFELINE_ABBR)))
        ax.set_xticklabels(LIFELINE_ABBR, rotation=0, ha='center', fontsize=12, fontweight='bold')
        ax.set_yticks(np.arange(len(indicator_order)))
        ax.set_yticklabels([f"{i:02d}" for i in range(1, len(indicator_order) + 1)], fontsize=10)

        for sep in [5.5, 11.5, 17.5, 23.5]:
            ax.axhline(sep, color='white', linewidth=1.2, alpha=0.9)

        # ax.set_xlabel('Community Lifeline (abbreviation)', fontsize=13)
        # ax.set_ylabel('Reasoning Indicator', fontsize=13)
        style_axis(ax, add_grid=False)

        cbar = fig.colorbar(im, ax=ax, fraction=0.023, pad=0.02)
        cbar.ax.tick_params(labelsize=10)

        out_path = OUTPUT_DIR / out_name
        fig.savefig(out_path, dpi=600, transparent=True, bbox_inches='tight')
        plt.show()
        print(f'Saved: {out_path}')

# Kappa heatmaps
_plot_heatmap_pair(
    model_mats=model_mats_kappa,
    out_name_q='Fig4_Heatmap_Qwen36.png',
    out_name_i='Fig4_Heatmap_InternVL.png',
    cmap=cmap_kappa,
    vmin=vmin_k,
    vmax=vmax_k,
    cbar_label='kappa_w',
)

# Agreement heatmaps
_plot_heatmap_pair(
    model_mats={m: model_mats_agree[m] * 100.0 for m in MODEL_ORDER},
    out_name_q='Fig4_Heatmap_Agreement_Qwen36.png',
    out_name_i='Fig4_Heatmap_Agreement_InternVL.png',
    cmap=cmap_agree,
    vmin=vmin_a,
    vmax=vmax_a,
    cbar_label='agreement_rate (%)',
)

In [ ]:
# 8) Final reporting block
main8_availability = availability.copy()
status_by_lifeline = main8_availability.groupby('Lifeline')['Status'].apply(lambda s: ', '.join(sorted(set(s)))).reset_index()

valid_kappa_total = int(main8_df['kappa_w'].notna().sum())
missing_kappa_total = int(main8_df['kappa_w'].isna().sum())

print('Run summary')
print('-----------')
print(f'Lifelines analyzed (main 8): {LIFELINE_ORDER}')
print('\nAvailability status by lifeline (combined across models):')
display(status_by_lifeline)
print(f'Valid kappa count (main 8): {valid_kappa_total}')
print(f'Missing/undefined kappa count (main 8): {missing_kappa_total}')

print('\nOverall performance by lifeline and model:')
display(overall_perf)

print('\nExported files:')
exports = [
    availability_path,
    overall_path,
    geo_path,
    ev_path,
    clean_long_path,
    OUTPUT_DIR / 'Fig1_Lifeline_Overall_Kappa.png',
    OUTPUT_DIR / 'Fig2_Lifeline_Reasoning_Evidence.png',
    OUTPUT_DIR / 'Fig3_Lifeline_Geographic_Cues.png',
    OUTPUT_DIR / 'Fig4_Heatmap_Qwen36.png',
    OUTPUT_DIR / 'Fig4_Heatmap_InternVL.png',
    OUTPUT_DIR / 'Fig4_Heatmap_Agreement_Qwen36.png',
    OUTPUT_DIR / 'Fig4_Heatmap_Agreement_InternVL.png',
]
for p in exports:
    print(f'- {p}')

## Brief Interpretation

Across Community Lifelines, reasoning performance varies by model and by reasoning component.

- Overall kappa levels are heterogeneous across Lifelines, indicating that geographic reasoning difficulty is Lifeline-dependent rather than uniform.
- Geographic Cue profiles differ across Lifelines, with some cue families showing stronger consistency than others.
- Reasoning Evidence dimensions also vary across Lifelines, suggesting that cue utilization strategies are not identical across operational contexts.

Missing values are treated as undefined observations (not poor performance), and no interpolation is applied. This notebook reports descriptive comparisons only and does not claim statistical significance or causality.